# Phase 3: Narratives (Assembled)
Runs all 6 LLM calls via NarrativeAssembler.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == 'certification_framework' else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import json
from scripts.narratives.assembler import NarrativeAssembler

NB_DATA = ROOT / 'notebooks' / 'data'
phase1_path = NB_DATA / 'parsed_context.json'
phase2_path = NB_DATA / 'computed_content.json'

assembler = NarrativeAssembler(phase1_path, phase2_path)
result = await assembler.assemble()

# Save to notebook-local data
(NB_DATA / 'narratives.json').write_text(
    json.dumps(result, indent=2, default=str, ensure_ascii=False), encoding='utf-8'
)

print(f'Keys: {list(result.keys())}')
print(f'Fallbacks used: {result.get("fallbacks_used", False)}')
print(f'Errors: {len(result.get("errors", []))}')

In [ ]:
print('=== Narrative Outputs ===')

# Scope narrative
if 'scope_narrative' in result:
    sn = result['scope_narrative']
    print(f'\nScope Narrative (source={sn.get("source","?")}):')
    print(f'  {sn.get("text","")[:200]}...')

# Key findings
if 'key_findings' in result:
    kf = result['key_findings']
    print(f'\nKey Findings ({len(kf.get("items",[]))} items):')
    for f in kf.get('items', []):
        print(f'  [{f.get("severity","")}] {f.get("headline","")}: {f.get("detail","")[:60]}')

# Qualitative findings
if 'qualitative_findings' in result:
    qf = result['qualitative_findings']
    dims = [k for k in qf if isinstance(qf[k], list)]
    print(f'\nQualitative Findings ({len(dims)} dimensions):')
    for dim in dims:
        print(f'  {dim}: {len(qf[dim])} findings')

# Fault analysis
if 'fault_category_analysis' in result:
    fa = result['fault_category_analysis']
    cats = fa.get('categories', {})
    print(f'\nFault Analysis ({len(cats)} categories):')
    for cat, analysis in cats.items():
        print(f'  {cat}: {str(analysis.get("detail",""))[:70]}')

# Limitations
if 'limitations_enriched' in result:
    le = result['limitations_enriched']
    print(f'\nLimitations ({len(le.get("items",[]))} items):')
    for item in le.get('items', []):
        print(f'  [{item.get("severity","")}] {item.get("limitation","")[:70]}')

# Recommendations
if 'recommendations_enriched' in result:
    re_ = result['recommendations_enriched']
    print(f'\nRecommendations ({len(re_.get("items",[]))} items):')
    for item in re_.get('items', []):
        print(f'  [{item.get("priority","")}] {item.get("recommendation","")[:70]}')